In [1]:
import os
from dotenv import load_dotenv

# Carica le variabili d'ambiente
load_dotenv()

# Importazioni necessarie
from datapizza.clients import (
    ClientFactory, 
    OpenAIClient,
    AnthropicClient, 
    GoogleClient,
    MistralClient,
    AzureOpenAIClient
)
from datapizza.clients.factory import Provider

In [4]:
import os
from datapizza.clients import ClientFactory

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5",
    system_prompt=(
        "Sei un assistente utile e conciso. "
        "Rispondi in italiano, usa elenchi puntati quando opportuno."
    ),
    temperature=1,
)

In [5]:
from datapizza.memory import Memory
from datapizza.type import TextBlock, ROLE

memory = Memory()

# Aggiunta di un turno utente
memory.add_turn([TextBlock(content="Ciao, sono Marco")], ROLE.USER)

# Invocazione con contesto
response = client.invoke("", memory=memory)
# Salvataggio risposta
memory.add_turn([TextBlock(content=response.text)], ROLE.ASSISTANT)

In [15]:
def chat_turn(user_input: str) -> str:
    memory.add_turn([TextBlock(content=user_input)], ROLE.USER)
    response = client.invoke("", memory=memory)
    memory.add_turn([TextBlock(content=response.text)], ROLE.ASSISTANT)
    return response.text

print(chat_turn("Presentati in una frase."))
print(chat_turn("Ora dimmi 3 best practice per Django."))

Sono un assistente virtuale in italiano, rapido e conciso, pronto ad aiutarti con risposte chiare e soluzioni pratiche.
- Configurazioni e segreti per ambiente
  - Separa settings per dev/staging/prod; usa variabili d’ambiente (es. django-environ) e non committare segreti.
  - DEBUG=False in produzione, ALLOWED_HOSTS impostato, logging configurato.
  - Gestisci SECRET_KEY, credenziali DB e chiavi API con un secret manager.

- Query/ORM e performance
  - Evita N+1 con select_related/prefetch_related; aggiungi indici (db_index, Index, UniqueConstraint).
  - Limita i campi (only/defer/values), usa paginazione e cache per viste/query frequenti.
  - Usa transazioni (atomic) e ottimizza migrazioni piccole e frequenti.

- Sicurezza e deploy
  - Attiva SecurityMiddleware, HTTPS con SECURE_SSL_REDIRECT e HSTS.
  - Cookie sicuri: SESSION_COOKIE_SECURE, CSRF_COOKIE_SECURE, HttpOnly, SameSite; protezione CSRF/XSS e X_FRAME_OPTIONS.
  - Aggiorna dipendenze, gestisci static con WhiteNoise, e usa un 

In [16]:
from typing import Optional

class Chatbot:
    def __init__(self, client, window_size: int = 6):
        self.client = client
        self.memory = Memory()
        self.window_size = window_size

    def _apply_sliding_window(self):
        if len(self.memory.memory) > self.window_size:
            self.memory.memory = self.memory.memory[-self.window_size:]

    def send(self, user_input: str) -> str:
        self.memory.add_turn([TextBlock(content=user_input)], ROLE.USER)
        self._apply_sliding_window()
        response = self.client.invoke("", memory=self.memory)
        self.memory.add_turn([TextBlock(content=response.text)], ROLE.ASSISTANT)
        return response.text

In [17]:
bot = Chatbot(client, window_size=6)
print("Scrivi 'esci' o 'exit' per terminare.")

while True:
    try:
        user = input("tu> ").strip()
        if user.lower() in {"esci", "exit", "quit"}:
            break
        answer = bot.send(user)
        print(f"bot> {answer}")
    except KeyboardInterrupt:
        break
    except Exception as e:
        print(f"errore> {e}")

Scrivi 'esci' o 'exit' per terminare.


tu>  Ciao, cosa sai fare?


bot> Ciao! Posso aiutarti in molti modi, ad esempio:

- Rispondere a domande e spiegare concetti (scuola, lavoro, curiosità)
- Riassumere o riformulare testi, prendere appunti, creare scalette
- Scrivere e migliorare email, CV, lettere, post, articoli
- Tradurre e rivedere testi in varie lingue
- Fare tutoraggio su materie scolastiche/universitarie
- Programmare e fare debug (Python, JavaScript, SQL, ecc.)
- Creare formule per Excel/Google Sheets e automatizzare attività
- Risolvere problemi di matematica e fare calcoli passo‑passo
- Analizzare dati a livello base: descrivere, pulire, interpretare risultati
- Ideare: brainstorming per progetti, contenuti, startup, nomi, slogan
- Pianificare: studio, allenamento, viaggi, menu, budget personali
- Preparare colloqui, presentazioni e discorsi
- Dare suggerimenti pratici e troubleshooting tecnico generale

Dimmi cosa ti serve e partiamo!


tu>  Quanto fa 65723657*148


bot> 9727101236


tu>  exit


In [21]:
from datapizza.cache import MemoryCache

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5",
    cache=MemoryCache(),  # supportato solo da OpenAI nel costruttore
    temperature=1
)

# Esempio di ispezione metrica
reply = client.invoke("Dammi 3 consigli per testare un'API REST")
print("token prompt:", reply.prompt_tokens_used)
print("token risposta:", reply.completion_tokens_used)
print("stop reason:", reply.stop_reason)

token prompt: 18
token risposta: 950
stop reason: stop


In [ ]:
import os
from datapizza.clients import ClientFactory
from datapizza.memory import Memory
from datapizza.type import TextBlock, ROLE

class Chatbot:
    def __init__(self, client, window_size: int = 6):
        self.client = client
        self.memory = Memory()
        self.window_size = window_size

    def _apply_sliding_window(self):
        if len(self.memory.memory) > self.window_size:
            self.memory.memory = self.memory.memory[-self.window_size:]

    def send(self, user_input: str) -> str:
        self.memory.add_turn([TextBlock(content=user_input)], ROLE.USER)
        self._apply_sliding_window()
        response = self.client.invoke("", memory=self.memory)
        self.memory.add_turn([TextBlock(content=response.text)], ROLE.ASSISTANT)
        # Stampa metriche minime (opzionale)
        total_tokens = (response.prompt_tokens_used or 0) + (response.completion_tokens_used or 0)
        print(f"[metriche] token totali: {total_tokens}")
        return response.text

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-4o",
    system_prompt=(
        "Sei un assistente utile e conciso. Rispondi in italiano "
        "e proponi sempre un breve elenco di prossimi passi."
    ),
    temperature=0.6,
)

bot = Chatbot(client, window_size=6)
print("Chat pronta. Digita 'esci' per terminare.")
while True:
    try:
        user = input("tu> ").strip()
        if user.lower() in {"esci", "exit", "quit"}:
            break
        print("bot>", bot.send(user))
    except KeyboardInterrupt:
        break
    except Exception:
        print("bot> Si è verificato un errore temporaneo. Riprova.")

Chat pronta. Digita 'esci' per terminare.


tu>  Ciao bello, chi sei?


[metriche] token totali: 119
bot> Ciao! Sono un assistente virtuale progettato per aiutarti con informazioni e risposte alle tue domande. Come posso aiutarti oggi?

Prossimi passi:
1. Fammi sapere di cosa hai bisogno.
2. Chiedimi qualsiasi informazione o supporto.
3. Approfitta delle mie capacità per risolvere i tuoi dubbi.


tu>  Sai chi è Mirko Calcaterra?


[metriche] token totali: 248
bot> Mi dispiace, non ho informazioni specifiche su Mirko Calcaterra. Potrebbe essere una persona non pubblicamente riconosciuta o conosciuta in un contesto specifico. Se hai ulteriori dettagli o un contesto specifico, potrei cercare di aiutarti meglio.

Prossimi passi:
1. Fornisci ulteriori dettagli o contesto su Mirko Calcaterra.
2. Specifica l'ambito di interesse (es. professionale, personale).
3. Chiedi ulteriori informazioni su argomenti correlati.


tu>  Dai quella che lavora in Datapizza, non puoi connetterti a Internet?


[metriche] token totali: 388
bot> Non ho la capacità di connettermi a Internet in tempo reale per ottenere informazioni aggiornate o specifiche su individui non pubblicamente noti. Tuttavia, posso aiutarti con informazioni generali o consigliarti su come trovare ciò che cerchi.

Prossimi passi:
1. Controlla i social media o LinkedIn per informazioni su Mirko Calcaterra.
2. Cerca "Datapizza" per vedere se ci sono notizie o contatti aziendali.
3. Se hai bisogno di altro supporto, fammi sapere!
